# Group Commit, and Why One Flush Serves Many Transactions

Companion notebook for the [Write Ahead Logging](https://codemia.io/courses/database_internals/dbi_write_ahead_logging) lesson on Codemia.

**What you will build**: an event driven model of the commit path, with a log device that has a
fixed flush cost and a finite bandwidth, and the throughput and latency that come out of it. By the
end you will be able to say why a database that can only flush two thousand times a second commits
fifty thousand transactions a second, where the ceiling actually is, and what a commit delay setting
buys and costs.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(3)

FLUSH_FIXED = 0.0005      # seconds a flush costs before any bytes are considered
BYTES_PER_TXN = 4096      # log bytes one transaction contributes
DEVICE_BPS = 1.0e9        # bytes per second the log device sustains
N = 20000                 # transactions per run

## Concept recap

A transaction may not report success until its log records are durable, and making bytes durable
means a flush to the device. A flush has two costs that behave completely differently: a fixed cost
per flush, which is the device latency, and a cost per byte, which is the device bandwidth.

If every transaction flushed on its own, throughput would be one over the fixed cost. A device that
takes half a millisecond per flush would cap the database at two thousand commits a second, and
that number would not improve with more cores, more clients or a faster processor.

Group commit removes that cap. While one flush is in progress, every transaction that reaches its
commit point queues up behind it, and when the flush finishes they all go into the next one. The
fixed cost is now paid once for the whole group instead of once per transaction, and the system
settles at a batch size that exactly absorbs the offered load. The remaining ceiling is bandwidth,
which is a different number entirely.

The cost is latency. A transaction that arrives just after a flush starts waits for that flush to
end before its own even begins, so its commit takes up to two flush times rather than one. That is
the trade, and a commit delay setting lets you push further along it deliberately.

## The commit path as an event loop

In [ ]:
def flush_time(batch):
    """A flush costs a fixed amount plus the bytes it carries."""
    return FLUSH_FIXED + batch * BYTES_PER_TXN / DEVICE_BPS


def run(rate, delay=0.0, group=True, n=N, seed=3):
    """Poisson arrivals at `rate` commits per second through one log device.

    Returns per transaction latency, the batch size of every flush, and the wall clock
    at which the last transaction completed.
    """
    gen = np.random.default_rng(seed)
    arrivals = np.cumsum(gen.exponential(1.0 / rate, size=n))
    latency = np.empty(n)
    batches = []

    i = 0            # first transaction not yet durable
    clock = 0.0      # the device is free at this time
    while i < n:
        start = max(clock, arrivals[i]) + delay
        if group:
            # everyone who reached their commit point before the flush starts rides along
            j = int(np.searchsorted(arrivals, start, side="right"))
            j = max(j, i + 1)
        else:
            j = i + 1
        batch = j - i
        clock = start + flush_time(batch)
        latency[i:j] = clock - arrivals[i:j]
        batches.append(batch)
        i = j

    return latency, np.array(batches), clock


def summary(rate, **kw):
    latency, batches, clock = run(rate, **kw)
    return {
        "offered": rate,
        "achieved": N / clock,
        "mean_ms": 1000 * latency.mean(),
        "p99_ms": 1000 * np.percentile(latency, 99),
        "batch": batches.mean(),
        "flushes": len(batches),
    }

In [ ]:
rates = [200, 500, 1000, 2000, 5000, 10000, 20000, 50000, 100000]
grouped = [summary(r) for r in rates]
alone = [summary(r, group=False) for r in rates]

print("group commit on")
print("%10s %10s %9s %9s %9s %9s" %
      ("offered", "achieved", "mean ms", "p99 ms", "per flush", "flushes"))
for s in grouped:
    print("%10d %10.0f %9.2f %9.2f %9.1f %9d" %
          (s["offered"], s["achieved"], s["mean_ms"], s["p99_ms"], s["batch"], s["flushes"]))

print()
print("one flush per transaction")
print("%10s %10s %9s %9s" % ("offered", "achieved", "mean ms", "p99 ms"))
for s in alone:
    print("%10d %10.0f %9.2f %9.2f" % (s["offered"], s["achieved"], s["mean_ms"], s["p99_ms"]))

The second table stops at about two thousand commits a second and stays there no matter what is
offered, and the mean latency climbs without limit once the offered load passes that point. That is
a queue growing, which in a real system is a connection pool filling up and then an application
timing out.

The first table keeps going well past it. Nothing was tuned to make that happen: the batch size
column grows on its own as load rises, because a batch is simply everyone who arrived during the
previous flush. Group commit is self regulating in exactly the way a fixed batch size would not be.

## The two ceilings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot([s["offered"] for s in grouped], [s["achieved"] for s in grouped], "o-",
             color="#5b8def", lw=2, label="group commit")
axes[0].plot([s["offered"] for s in alone], [s["achieved"] for s in alone], "o-",
             color="#d1495b", lw=2, label="one flush each")
axes[0].axhline(1 / FLUSH_FIXED, color="#d1495b", ls="--", lw=1)
axes[0].axhline(DEVICE_BPS / BYTES_PER_TXN, color="#5b8def", ls="--", lw=1)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("offered commits per second")
axes[0].set_ylabel("achieved commits per second")
axes[0].set_title("the dashed lines are 1/latency and bandwidth/bytes")
axes[0].legend()

axes[1].plot([s["offered"] for s in grouped], [s["mean_ms"] for s in grouped], "o-",
             color="#5b8def", lw=2, label="mean")
axes[1].plot([s["offered"] for s in grouped], [s["p99_ms"] for s in grouped], "o-",
             color="#8a5bd6", lw=2, label="99th percentile")
axes[1].set_xscale("log")
axes[1].set_xlabel("offered commits per second")
axes[1].set_ylabel("commit latency, milliseconds")
axes[1].set_title("group commit latency")
axes[1].legend()

plt.tight_layout()
plt.show()

print("flush latency ceiling   %8.0f commits per second" % (1 / FLUSH_FIXED))
print("device bandwidth ceiling %7.0f commits per second" % (DEVICE_BPS / BYTES_PER_TXN))

Two horizontal lines, two different ceilings, and more than two orders of magnitude between them.

Without batching the system lives against the lower one, which is a property of how long a single
flush takes. With batching it lives against the upper one, which is a property of how many bytes the
device can absorb per second divided by how many bytes a transaction writes.

That second form is the useful one, because both of its terms are things you can change. Fewer bytes
per transaction raises the ceiling directly, which is why log record size is a performance concern
and not only a storage one. A device with more bandwidth raises it too, while a device with lower
latency and the same bandwidth changes almost nothing once batching is working.

## What a commit delay buys

In [ ]:
print("%10s %10s %9s %9s %10s %9s" %
      ("delay ms", "achieved", "mean ms", "p99 ms", "per flush", "flushes"))
for d in (0.0, 0.0002, 0.0005, 0.001, 0.002, 0.005):
    s = summary(5000, delay=d)
    print("%10.1f %10.0f %9.2f %9.2f %10.1f %9d" %
          (1000 * d, s["achieved"], s["mean_ms"], s["p99_ms"], s["batch"], s["flushes"]))

A commit delay makes every transaction wait a fixed amount before its flush begins, so that more of
them accumulate. The batch grows, the number of flushes falls, and latency rises by roughly the
delay. At this offered load that is a bad bargain: throughput was never the constraint, so the delay
buys fewer device operations and pays for them in latency that every client sees.

It becomes a good bargain in exactly one situation, which is when the device is the constraint and
the flushes are small. Then the fixed cost per flush is a real fraction of the work, and trading a
fraction of a millisecond of latency for half as many flushes is worth doing. That is why the
setting exists, why it defaults to zero, and why turning it on without measuring the batch size
first is a reliable way to make a database slower.

## Variations: three durability settings on one device

In [ ]:
RATE = 5000

per_commit = summary(RATE, group=False)
grouped_now = summary(RATE)

# Asynchronous commit runs the same device and the same batching. The difference is that
# the transaction does not wait for the flush, so its commit latency is the cost of
# appending to an in memory buffer, and the exposure is whatever has not been flushed yet.
async_latency_ms = 0.0
async_at_risk_s = 1.0

rows = [
    ("every commit", per_commit["achieved"], per_commit["mean_ms"], 0.0),
    ("group commit", grouped_now["achieved"], grouped_now["mean_ms"], 0.0),
    ("asynchronous", grouped_now["achieved"], async_latency_ms, async_at_risk_s),
]

print("offered load %d commits per second" % RATE)
print("%-14s %10s %10s %18s" % ("setting", "achieved", "mean ms", "writes at risk"))
for name, achieved, ms, risk in rows:
    print("%-14s %10.0f %10.2f %14.0f sec" % (name, achieved, ms, risk))
print()
print("the third row achieves what the second does because the device keeps up at this load,")
print("and it differs from the second only in what a crash would cost")

The three rows are the three answers a database can give to the question of when a commit is safe,
and only the third one is a different guarantee rather than a different performance point.

Flushing per transaction and group commit both promise that an acknowledged transaction survives a
crash. They differ in how that promise is implemented and therefore in throughput, and the honest
summary is that group commit is strictly better: the same guarantee, more throughput, slightly more
latency under light load.

Asynchronous commit is not on that scale. It returns before the bytes are durable, so a crash loses
whatever had not been flushed. That is a legitimate choice for a table of page view counters and a
poor one for a ledger, and the only mistake is making it without knowing which of the two you have.

## What to try next

- Vary the bytes per transaction from 256 to 64 kilobytes and plot where the bandwidth ceiling lands.
  A wide row or a table with many indexes moves this more than most people expect.
- Add a second device for the log with a different fixed cost, say a network attached volume at four
  milliseconds, and see how much of the gap batching closes.
- Model a flush that occasionally takes ten times as long, which is what a device under garbage
  collection does, and look at what it does to the 99th percentile rather than the mean.
- Make the arrivals bursty rather than Poisson and compare the batch size distribution. Group commit
  benefits from bursts in a way that a fixed batch size never could.